<a href="https://colab.research.google.com/github/Preetitamrakar-phd/GenAI_Hands-on/blob/main/Indexing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Indexing Strategies for RAG

How do we ORGANIZE documents for efficient retrieval

THE 5 STRATEGIES WE'LL EXPLORE:

1. Vector Index (Flat)     - Embed everything, similarity search
2. Summary Index           - Store full docs, LLM evaluates relevance
3. Tree Index              - Hierarchical: summaries → details
4. Keyword Table Index     - Traditional inverted index
5. Hybrid Retrieval        - Combine vector + keyword



In [1]:
'''
WHEN TO USE WHAT:
                    Vector    Summary   Tree      Keyword   Hybrid
Small dataset       ✓ Good    ✓ Good    Overkill  ✓ Good    Overkill
Large dataset       ✓ Best    ✗ Slow    ✓ Good    ✓ Fast    ✓ Best
Semantic queries    ✓ Best    ✓ Good    ✓ Good    ✗ Bad     ✓ Best
Exact match (IDs)   ✗ Bad     ✗ Bad     ✗ Bad     ✓ Best    ✓ Good
Hierarchical docs   ✗ Bad     ✓ OK      ✓ Best    ✗ Bad     ✓ Good
'''


'\nWHEN TO USE WHAT:\n                    Vector    Summary   Tree      Keyword   Hybrid\nSmall dataset       ✓ Good    ✓ Good    Overkill  ✓ Good    Overkill\nLarge dataset       ✓ Best    ✗ Slow    ✓ Good    ✓ Fast    ✓ Best\nSemantic queries    ✓ Best    ✓ Good    ✓ Good    ✗ Bad     ✓ Best\nExact match (IDs)   ✗ Bad     ✗ Bad     ✗ Bad     ✓ Best    ✓ Good\nHierarchical docs   ✗ Bad     ✓ OK      ✓ Best    ✗ Bad     ✓ Good\n'

FRAMEWORK: LlamaIndex

This module uses LlamaIndex (not LangChain) because it has
excellent built-in support for different indexing strategies.

LangChain excels at: Chains, agents, integrations
LlamaIndex excels at: Document indexing, retrieval patterns

In [7]:
import json
import os
import httpx
from dotenv import load_dotenv

In [8]:
# LlamaIndex core components
from llama_index.core import (
    VectorStoreIndex,    # Standard embedding-based index
    SummaryIndex,        # Full document storage, LLM-based relevance
    TreeIndex,           # Hierarchical summarization tree
    KeywordTableIndex,   # Inverted keyword index
    Document,            # Document wrapper with text + metadata
    Settings             # Global configuration
)
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

In [4]:
#!pip install llama_index


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 14.7 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, 

In [15]:
# SETUP: Load Environment Variables
from google.colab import files
uploaded = files.upload()

Saving synthetic_tickets.json to synthetic_tickets.json


In [11]:
load_dotenv()

True

In [12]:
# Set longer timeout for httpx (used by OpenAI client)
# Some index types make MANY LLM calls and need more time
os.environ["HTTPX_TIMEOUT"] = "300"  # 5 minutes


 CONFIGURE LLAMAINDEX SETTINGS


 LlamaIndex uses a Settings singleton to configure:
   - embed_model: Which embedding model to use
   - llm: Which LLM to use for queries and index building

These settings apply globally to all indexes we create.

In [13]:
Settings.embed_model = OpenAIEmbedding(
    model=os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small'),
    api_key=os.getenv('OPENAI_API_KEY'),
    timeout=120,      # 2 min timeout for embedding calls
    max_retries=5     # Retry on failure
)
Settings.llm = OpenAI(
    model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
    api_key=os.getenv('OPENAI_API_KEY'),
    timeout=300,      # 5 min timeout (Tree/Keyword indexes are slow!)
    max_retries=5
)

In [14]:
print("INDEXING STRATEGIES FOR RAG")
print("1. Vector Index - Semantic similarity search (MOST COMMON)")
print("2. Summary Index - Search through full documents with LLM")
print("3. Tree Index - Hierarchical retrieval (summaries → details)")
print("4. Keyword Table Index - Traditional keyword matching")
print("5. Hybrid Retrieval - Combine multiple strategies (PRODUCTION)")

INDEXING STRATEGIES FOR RAG
1. Vector Index - Semantic similarity search (MOST COMMON)
2. Summary Index - Search through full documents with LLM
3. Tree Index - Hierarchical retrieval (summaries → details)
4. Keyword Table Index - Traditional keyword matching
5. Hybrid Retrieval - Combine multiple strategies (PRODUCTION)


In [18]:
# LOAD DATA
print("Loading support Tickets")

with open('synthetic_tickets.json', 'r', encoding='utf-8') as f:
    tickets = json.load(f)

Loading support Tickets


Convert to LlamaIndex Documents

 LlamaIndex uses Document objects (similar to LangChain's Document)

 Each Document has:
   - text: The content to index
   - metadata: Associated data for filtering/display

In [26]:
documents = []
for ticket in tickets:
    # Combine all fields into content (rich context for embedding)
    # IMPORTANT: Include ticket_id in text so keyword index can find it!
    content = f"""Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Description: {ticket['description']}
Resolution: {ticket['resolution']}
Category: {ticket['category']}
Priority: {ticket['priority']}"""

    doc = Document(
        text=content,
        metadata={
            'ticket_id': ticket['ticket_id'],
            'category': ticket['category'],
            'priority': ticket['priority'],
            'title': ticket['title']
        }
    )
    documents.append(doc)

print(f"✓ Loaded {len(documents)} support tickets")


✓ Loaded 20 support tickets


In [27]:
query = "How do I fix authentication issues after password reset?"
print(f"\nTest Query: '{query}'")


Test Query: 'How do I fix authentication issues after password reset?'


PART 1: Vector Index (Flat Index)

 HOW IT WORKS:

   Build time:

     Document → [Chunk] → [Embed] → Store vector in index

   Query time:
   
     Query → [Embed] → [Find nearest vectors] → Return top-K documents

In [28]:
# Build the Vector Index
# -----------------------------------------------------------------------------
# from_documents() handles:
#   1. Chunking (if needed - our docs are small so no chunking)
#   2. Embedding each chunk via Settings.embed_model
#   3. Storing vectors in memory (or a vector store if configured)
# -----------------------------------------------------------------------------
vector_index = VectorStoreIndex.from_documents(documents)

In [29]:
# Create Query Engine
# -----------------------------------------------------------------------------
# as_query_engine() wraps the index for easy querying:
#   - similarity_top_k=3: Return top 3 most similar documents
#   - The engine handles: embed query → search → synthesize response
# -----------------------------------------------------------------------------
vector_query_engine = vector_index.as_query_engine(similarity_top_k=3)

In [32]:
print("✓ Created vector index")
print(f"\nQuery: '{query}'")
vector_response = vector_query_engine.query(query)


✓ Created vector index

Query: 'How do I fix authentication issues after password reset?'


In [33]:
print("\nVector Index Results:")
print(f"Answer: {vector_response.response}\n")
print("Source Documents:")
for i, node in enumerate(vector_response.source_nodes, 1):
    print(f"\n{i}. {node.metadata.get('ticket_id', 'Unknown')}")
    print(f"   Score: {node.score:.4f}")  # Similarity score (higher = more similar)
    print(f"   {node.text[:150]}...")


Vector Index Results:
Answer: To fix authentication issues after a password reset, clear all active sessions and force re-authentication for users. Additionally, implement automatic session cleanup whenever a password is changed to prevent similar issues in the future.

Source Documents:

1. TICK-001
   Score: 0.6259
   Ticket ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing ...

2. TICK-011
   Score: 0.4343
   Ticket ID: TICK-011
Title: SSO authentication broken after upgrade
Description: Single Sign-On integration with corporate SAML provider failing. Users...

3. TICK-016
   Score: 0.4181
   Ticket ID: TICK-016
Title: Two-factor authentication codes not working
Description: Users unable to log in with TOTP codes from authenticator apps. Co...


PART 2: Summary Index

 HOW IT WORKS:

   Build time:

     Just store all documents as-is (no embedding!)

   Query time:

     For EACH document, ask LLM: "Is this relevant to the query?"
     
     Collect relevant docs → Synthesize answer

In [34]:
summary_index = SummaryIndex.from_documents(documents)

In [35]:
summary_query_engine = summary_index.as_query_engine(response_mode="tree_summarize")

print("✓ Created summary index")
print(f"\nQuery: '{query}'")
summary_response = summary_query_engine.query(query)

print("\nSummary Index Results:")
print(f"Answer: {summary_response.response}\n")
print("Source Documents:")
for i, node in enumerate(summary_response.source_nodes[:3], 1):
    print(f"\n{i}. {node.metadata.get('ticket_id', 'Unknown')}")
    print(f"   {node.text[:150]}...")

✓ Created summary index

Query: 'How do I fix authentication issues after password reset?'

Summary Index Results:
Answer: To resolve authentication issues after a password reset, clear all active sessions and force users to re-authenticate. Additionally, implement automatic session cleanup whenever a password is changed to prevent similar issues in the future.

Source Documents:

1. TICK-001
   Ticket ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing ...

2. TICK-002
   Ticket ID: TICK-002
Title: Database connection timeout in production
Description: Application experiencing intermittent 500 errors. Logs show 'connect...

3. TICK-003
   Ticket ID: TICK-003
Title: Payment processing fails for international cards
Description: Customers using non-US credit cards receiving 'Payment declin...


# PART 3: Tree Index (Hierarchical Retrieval)

In [36]:
# HOW THE TREE IS BUILT (Bottom-Up, EXPENSIVE):
# ──────────────────────────────────────────────
#   1. Each document/chunk becomes a LEAF node
#   2. Leaves are grouped SEQUENTIALLY (by insertion order!) into
#      groups of `num_children` — NOT by topic similarity!
#   3. LLM summarizes each group → creates parent SUMMARY nodes
#   4. Repeat on summaries until a single ROOT node remains

# HOW QUERIES WORK (EFFICIENT - log(n) traversal):
# ─────────────────────────────────────────────────
#   1. Start at root
#   2. LLM scores all children: "Which branch is most relevant?"
#   3. Follow top `child_branch_factor` branches
#   4. Repeat until reaching leaf nodes
#   5. Collect leaves, synthesize final answer

# TRADE-OFF:
#   Build time: SLOW (many LLM calls to create summaries)
#   Query time: FAST (logarithmic traversal)

In [37]:
tree_documents = documents
print(f"Building Tree Index with {len(tree_documents)} documents...")

tree_index = TreeIndex.from_documents(tree_documents)
tree_query_engine = tree_index.as_query_engine(child_branch_factor=2)

print("✓ Created tree index with hierarchical structure")
print(f"\nQuery: '{query}'")

tree_response = tree_query_engine.query(query)

print("\nTree Index Results:")
print(f"Answer: {tree_response.response}\n")
print("Source Documents:")
for i, node in enumerate(tree_response.source_nodes[:3], 1):
    print(f"\n{i}. {node.metadata.get('ticket_id', 'Unknown')}")
    print(f"   {node.text[:150]}...")

Building Tree Index with 20 documents...
✓ Created tree index with hierarchical structure

Query: 'How do I fix authentication issues after password reset?'

Tree Index Results:
Answer: To fix authentication issues after a password reset, clear all active sessions and force users to re-authenticate. Additionally, implement automatic session cleanup whenever a password is changed to prevent similar issues in the future.

Source Documents:

1. TICK-001
   Ticket ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing ...

2. TICK-016
   Ticket ID: TICK-016
Title: Two-factor authentication codes not working
Description: Users unable to log in with TOTP codes from authenticator apps. Co...


PART 4: Keyword Table Index

In [38]:
# HOW IT WORKS:
# ─────────────
#   Build time:
#     For each document, extract keywords (via LLM or rules)
#     Build inverted index: keyword → [doc_ids]
#
#   Query time:
#     Extract keywords from query
#     Look up documents containing those keywords
#     Return matching documents

In [39]:
# Use all documents
keyword_documents = documents
print(f"Building Keyword Index with {len(keyword_documents)} documents...")

keyword_index = KeywordTableIndex.from_documents(keyword_documents)

# Show the extracted keyword table (inverted index)
keyword_table = keyword_index.index_struct.table
print(f"\n✓ Extracted {len(keyword_table)} unique keywords:")
for keyword, node_ids in sorted(keyword_table.items()):
    print(f"  '{keyword}' → {len(node_ids)} document(s)")

# Create query engine
keyword_query_engine = keyword_index.as_query_engine()

print("\n✓ Created keyword table index")
print(f"\nQuery: '{query}'")

Building Keyword Index with 20 documents...

✓ Extracted 236 unique keywords:
  '17' → 1 document(s)
  '413' → 1 document(s)
  '413 error' → 1 document(s)
  '8' → 1 document(s)
  'acceptance' → 1 document(s)
  'acceptance window' → 1 document(s)
  'accounts' → 1 document(s)
  'accuracy' → 1 document(s)
  'aggressive' → 1 document(s)
  'alerts' → 1 document(s)
  'algorithm' → 1 document(s)
  'anonymization' → 1 document(s)
  'api' → 2 document(s)
  'apis' → 1 document(s)
  'app' → 2 document(s)
  'app servers' → 1 document(s)
  'apps' → 1 document(s)
  'async' → 1 document(s)
  'async/await' → 1 document(s)
  'authenticated' → 1 document(s)
  'authentication' → 3 document(s)
  'authenticator' → 1 document(s)
  'authenticator apps' → 1 document(s)
  'await' → 1 document(s)
  'background' → 1 document(s)
  'background worker' → 1 document(s)
  'backlog' → 1 document(s)
  'backup' → 1 document(s)
  'balancer' → 1 document(s)
  'batch' → 1 document(s)
  'batch processing' → 1 document(s)
  

In [40]:
keyword_response = keyword_query_engine.query(query)

print("\nKeyword Index Results:")
print(f"Answer: {keyword_response.response}\n")
print("Source Documents:")
for i, node in enumerate(keyword_response.source_nodes[:3], 1):
    print(f"\n{i}. {node.metadata.get('ticket_id', 'Unknown')}")
    print(f"   {node.text[:150]}...")


Keyword Index Results:
Answer: To resolve authentication issues after a password reset, clear all active sessions and force users to re-authenticate. Additionally, implement automatic session cleanup whenever a password is changed to prevent similar issues in the future.

Source Documents:

1. TICK-001
   Ticket ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing ...

2. TICK-006
   Ticket ID: TICK-006
Title: Email notifications not being delivered
Description: Users reporting they're not receiving password reset and confirmation ...

3. TICK-011
   Ticket ID: TICK-011
Title: SSO authentication broken after upgrade
Description: Single Sign-On integration with corporate SAML provider failing. Users...


PART 5: Hybrid Retrieval

In [ ]:
# THE PROBLEM:
# ────────────
#   Vector search alone: Finds "login issues" for "auth problems" ✓
#                        Misses "Ticket T-123" for "T-123" ✗
#
#   Keyword search alone: Finds "Ticket T-123" for "T-123" ✓
#                         Misses "auth problems" for "login issues" ✗
#
# THE SOLUTION - Combine both:
# ────────────────────────────
#   Query: "authentication timeout error T-123"

In [ ]:
# FUSION STRATEGIES:
# ──────────────────
# 1. Simple union (what we do here) - combine and deduplicate
# 2. Reciprocal Rank Fusion (RRF) - score by rank in each list
#      score(doc) = Σ 1/(k + rank_i) for each retriever
# 3. Weighted combination - assign weights to each retriever

In [41]:
print("✓ Using Vector + Keyword hybrid approach")
print(f"\nQuery: '{query}'")

# Step 1: Retrieve from Vector Index (Semantic)
vector_nodes = vector_index.as_retriever(similarity_top_k=5).retrieve(query)

# Step 2: Retrieve from Keyword Index (Exact)
keyword_nodes = keyword_index.as_retriever().retrieve(query)

# Step 3: Fusion - Combine and Deduplicate
seen_ids = set()
hybrid_nodes = []

for node in vector_nodes + keyword_nodes:
    node_id = node.metadata.get('ticket_id', node.node_id)
    if node_id not in seen_ids:
        seen_ids.add(node_id)
        hybrid_nodes.append(node)

# Documents found by BOTH methods are likely most relevant!

print("\nHybrid Retrieval Results (Combined):")
for i, node in enumerate(hybrid_nodes[:3], 1):
    print(f"\n{i}. {node.metadata.get('ticket_id', 'Unknown')}")
    if hasattr(node, 'score') and node.score:
        print(f"   Score: {node.score:.4f}")
    print(f"   {node.text[:150]}...")


✓ Using Vector + Keyword hybrid approach

Query: 'How do I fix authentication issues after password reset?'

Hybrid Retrieval Results (Combined):

1. TICK-001
   Score: 0.6259
   Ticket ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing ...

2. TICK-011
   Score: 0.4343
   Ticket ID: TICK-011
Title: SSO authentication broken after upgrade
Description: Single Sign-On integration with corporate SAML provider failing. Users...

3. TICK-016
   Score: 0.4181
   Ticket ID: TICK-016
Title: Two-factor authentication codes not working
Description: Users unable to log in with TOTP codes from authenticator apps. Co...


PART 6: Comparison Summary

In [42]:
print("""
┌────────────────────┬───────────────────────────────┬──────────┬───────────┐
│ Index Type         │ Best For                      │ Speed    │ Accuracy  │
├────────────────────┼───────────────────────────────┼──────────┼───────────┤
│ Vector Index       │ General semantic search       │ Fast     │ High      │
│ Summary Index      │ High-level queries (small)    │ Slow     │ Medium    │
│ Tree Index         │ Large docs, hierarchical      │ Medium   │ High      │
│ Keyword Index      │ Exact keyword matching        │ Fast     │ Medium    │
│ Hybrid Retrieval   │ Production systems            │ Medium   │ Highest   │
└────────────────────┴───────────────────────────────┴──────────┴───────────┘

DECISION FLOWCHART:
───────────────────
                    ┌─────────────────────┐
                    │ What's your use case?│
                    └──────────┬──────────┘
                               │
           ┌───────────────────┼───────────────────┐
           ▼                   ▼                   ▼
    Small dataset       Large dataset        Production
    (<100 docs)         (1000s+ docs)        (any size)
           │                   │                   │
           ▼                   ▼                   ▼
    Vector Index          Tree Index         Hybrid
    (simple, fast)     (hierarchical)    (Vector + Keyword)

RECOMMENDATIONS:
────────────────
1. START with Vector Index - Works well for 90% of use cases
2. ADD Keyword Index for specific terminology/codes (error codes, IDs)
3. USE Tree Index for very large document collections (1000s+)
4. COMBINE Vector + Keyword for production (Hybrid)
5. AVOID Summary Index for large datasets (doesn't scale)

PRODUCTION BEST PRACTICE:
─────────────────────────
Vector Index + Keyword Index + Reciprocal Rank Fusion (RRF)
""")


┌────────────────────┬───────────────────────────────┬──────────┬───────────┐
│ Index Type         │ Best For                      │ Speed    │ Accuracy  │
├────────────────────┼───────────────────────────────┼──────────┼───────────┤
│ Vector Index       │ General semantic search       │ Fast     │ High      │
│ Summary Index      │ High-level queries (small)    │ Slow     │ Medium    │
│ Tree Index         │ Large docs, hierarchical      │ Medium   │ High      │
│ Keyword Index      │ Exact keyword matching        │ Fast     │ Medium    │
│ Hybrid Retrieval   │ Production systems            │ Medium   │ Highest   │
└────────────────────┴───────────────────────────────┴──────────┴───────────┘

DECISION FLOWCHART:
───────────────────
                    ┌─────────────────────┐
                    │ What's your use case?│
                    └──────────┬──────────┘
                               │
           ┌───────────────────┼───────────────────┐
           ▼                   ▼     